In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("ps04.ipynb")

# PS4 — The Phase Plane, Bistability, and the Toggle Switch
### BioE 147/247 · Fall 2026

**Out:** Thursday, September 24 · **Due:** Thursday, October 1, 11:59 pm
**Covers:** Sessions 8 and 9 · **42 points** · **Question 6** is required for **BioE 247** (48 total) and is extra credit for **BioE 147** (up to +6)

---

Two sessions, one object. Session 8 drew the phase plane and found where a
two-gene circuit can sit still and whether it stays there; session 9 turned one
knob on the device Gardner built and watched a state die. Questions 1–3 are the
first; 4–6 are the second.

Two of these questions are the items Tuesday's handout said would be here. They
are the same items.

**Notation is the paper's**, as in both sessions:

$$\frac{du}{dt} = \frac{\alpha_1}{1+v^{\beta}} - u
\qquad\qquad
\frac{dv}{dt} = \frac{\alpha_2}{1+u^{\gamma}} - v$$

`posb.toggle_model(alpha1, alpha2, n, m)` calls the two cooperativities `n`
(on $u$, in $dv/dt$) and `m` (on $v$, in $du/dt$), so $\gamma$ = `n` and
$\beta$ = `m`. The symmetric toggle has $\alpha_1 = \alpha_2 = a$ and
$\beta = \gamma = n$.

**Collaboration is encouraged.** Discuss, argue, work at a whiteboard together,
then write your own solution and your own code. Record who you worked with
below.

**If you used an LLM**, say so briefly and say what for. The conditions are that
you can explain anything you submit and that the code you submit runs.

**A note on the visible tests.** They check *properties* — symmetries, counts,
what happens at a limit. A green visible check means "not obviously broken",
not "right".

In [ ]:
COLLABORATORS = ""   # e.g. "worked with J. Chen on Q4"
AI_USE = ""          # e.g. "used an LLM to check the derivative in Q2"

## Setup

In [ ]:
# ---------------------------------------------------------------------------
# SETUP — run this cell first, every time.
#
# DataHub / local : finds the repository root and puts it on the import path.
# Google Colab    : clones the repository first, because Colab opens this
#                   notebook on its own, without the posb package beside it.
# ---------------------------------------------------------------------------
import os
import sys

if "google.colab" in sys.modules:
    if not os.path.exists("posb2026"):
        !git clone -q https://github.com/ArkinLaboratory/posb2026.git
    sys.path.insert(0, os.path.abspath("posb2026"))
else:
    _d = os.getcwd()
    while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, "posb")):
        _d = os.path.dirname(_d)
    sys.path.insert(0, _d)

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

import posb
from posb import Reaction, Model

posb.check_environment()


In [ ]:
from posb import (toggle_model, nullcline, fixed_points, jacobian,
                  classify, stability_report, toggle_alpha_critical)
from scipy.optimize import brentq

---
## Question 1 — Where can it sit still?

*T18, T19. Session 8, Runs 1 and 2.*

The symmetric toggle at $n = 2$, $a = 3$ — the example the session worked.
Everything here is the drawing, then the algebra, then the root finder, in that
order.

**Q1a.** Write `toggle_nullclines(a, n, grid)` returning the pair
`(u_of_v, v_of_u)`: the $u$-nullcline as $u$ evaluated at each `grid` value of
$v$, and the $v$-nullcline as $v$ at each `grid` value of $u$. Use
`posb.nullcline` — it is the same call the session's figures were made with —
or the closed form; the tests accept either. Then plot both on one pair of
axes with $u$ across and $v$ up.

In [ ]:
def toggle_nullclines(a, n, grid):
    """(u along the u-nullcline at each grid v, v along the v-nullcline at each grid u)."""
    ...


grid = np.linspace(0.001, 3.5, 400)
u_of_v, v_of_u = toggle_nullclines(3.0, 2, grid)
plt.figure(figsize=(4.5, 4.5))
plt.plot(u_of_v, grid, label="du/dt = 0")
plt.plot(grid, v_of_u, label="dv/dt = 0")
plt.xlabel("u"); plt.ylabel("v"); plt.gca().set_aspect("equal"); plt.legend()
plt.title("n = 2, a = 3");

In [ ]:
grader.check("q1a")

**Q1b.** Two functions.

`symmetric_root(a, n)`: the crossing on the diagonal, from
$x + x^{n+1} = a$, found with `brentq`. **Bracket it** — the left side is zero
at $x = 0$ and exceeds $a$ at $x = a$, so the bracket is on the page before
you write any code.

`all_fixed_points(a, n)`: every crossing, as a list of `(u, v)` tuples sorted
by $u$. Use `posb.fixed_points` with a small grid of starting guesses (a
$5 \times 5$ `np.geomspace` grid over $[0.01, 10]$ is plenty) — it is
`fsolve` from many starts, deduplicated. It finds the saddle too, which
forward integration never can.

In [ ]:
def symmetric_root(a, n):
    """x on the diagonal u = v = x, from x + x**(n+1) = a."""
    ...


def all_fixed_points(a, n):
    """Every fixed point of the symmetric toggle, [(u, v), ...] sorted by u."""
    ...


print(f"diagonal root at n = 2, a = 3: x = {symmetric_root(3.0, 2):.4f}")
for u, v in all_fixed_points(3.0, 2):
    print(f"  ({u:.4f}, {v:.4f})")

In [ ]:
grader.check("q1b")

<!-- BEGIN QUESTION -->

**Q1c.** *(written)*

1. `brentq` refuses to run without a bracket, and `fsolve` runs happily from
   any start and returns *a* root. Say in two sentences what the drawing in Q1a
   supplies that neither solver can supply for itself.
2. Why is there always a crossing on the diagonal, and why do the other two
   come as a mirror pair? Answer from the symmetry of the equations, not from
   the numbers.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 2 — Nudge it. Does it come back?

*T20. Session 8, Runs 3a and 3b.*

The Jacobian is four numbers, evaluated at a fixed point. Session 8 computed
them on the diagonal. Here you compute them anywhere.

**Q2a.** Write `toggle_jacobian(u, v, a1, a2, beta, gamma)` returning the
$2 \times 2$ Jacobian of the (possibly asymmetric) toggle at the point $(u, v)$
as a NumPy array, **analytically** — differentiate the right-hand sides by hand
and code the result. The tests compare it with `posb.jacobian`, which does it
by finite differences, at points that are and are not fixed points; the two
must agree everywhere, because a Jacobian is a property of the vector field,
not of the fixed point.

In [ ]:
def toggle_jacobian(u, v, a1, a2, beta, gamma):
    """d(du/dt, dv/dt) / d(u, v), analytically."""
    ...


x = symmetric_root(3.0, 2)
print("J on the diagonal, n = 2, a = 3:")
print(np.round(toggle_jacobian(x, x, 3.0, 3.0, 2, 2), 4))
print("eigenvalues:", np.round(np.linalg.eigvals(toggle_jacobian(x, x, 3.0, 3.0, 2, 2)), 4))

In [ ]:
grader.check("q2a")

**Q2b.** Write `classify_all(a, n)` returning a list of
`(u, v, label)` for every fixed point of the symmetric toggle, sorted by $u$,
with `label` one of `"stable"`, `"saddle"`, `"unstable"` — decided from the
**eigenvalues of your Q2a Jacobian**, not from `posb.classify`. (You may use
`posb.classify` to check yourself.)

In [ ]:
def classify_all(a, n):
    """[(u, v, 'stable' | 'saddle' | 'unstable'), ...] sorted by u."""
    ...


for u, v, lab in classify_all(3.0, 2):
    print(f"({u:.3f}, {v:.3f})  {lab}")

In [ ]:
grader.check("q2b")

<!-- BEGIN QUESTION -->

**Q2c.** *(written)* At the diagonal crossing of the $n = 2$, $a = 3$ toggle
your Jacobian is symmetric. Write down its two eigenvectors without computing
anything, say which eigenvalue goes with which, and say — in terms of the two
proteins — what a perturbation along the unstable one *is*.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 3 — n = 4 and n = 1

*T21. Session 9's faded set, finished properly.*

**Q3a.** Write `n_stable_states(a, n)` returning the number of *stable*
fixed points of the symmetric toggle. Then sweep $a$ over
`np.geomspace(0.3, 30, 25)` for $n = 1$ and for $n = 4$ and plot the count
against $a$ on a log axis, both on one figure.

In [ ]:
def n_stable_states(a, n):
    """How many stable fixed points does the symmetric toggle have?"""
    ...


a_grid = np.geomspace(0.3, 30, 25)
plt.figure(figsize=(6, 3))
for n in (1, 4):
    plt.plot(a_grid, [n_stable_states(a, n) for a in a_grid], "o-", label=f"n = {n}")
plt.xscale("log"); plt.xlabel("a"); plt.ylabel("stable states"); plt.legend();

In [ ]:
grader.check("q3a")

<!-- BEGIN QUESTION -->

**Q3b.** *(written)* Your $n = 4$ curve steps from one state to two at
$a_c(4) = 1.013$, and your $n = 1$ curve never steps at all. Session 8 derived
the first number for the symmetric toggle. Tuesday's handout proved something
stronger about the second. State that stronger result in one sentence, and say
why the symmetric derivation alone could not have given it to you.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 4 — pIKE105, the one that did not switch

*T22, T23. This is item 3 of the session 8 handout.*

Gardner built six toggle variants — four pTAK, two pIKE — that differ only
in the ribosome-binding site in front of *lacI* (RBS1 in Fig. 3). All four
pTAK are bistable; of the two pIKE only pIKE107 is. pIKE105 is not, and the
paper says why: TetR is a weaker repressor than cI, so the P$_\text{LtetO-1}$
arm had to be weakened relative to P$_\text{Ls1con}$, and in pIKE105 it was
not weakened enough.

We do not have pIKE105's parameters. We have pTAK117's, from the Fig. 5
legend — $\alpha_1 = 156.25$, $\alpha_2 = 15.6$, $\beta = 2.5$, $\gamma = 1$ —
and the question is the same shape. Gardner's swap moved $\alpha_1$; here you
hold $\alpha_1$ fixed and ask for which $\alpha_2$ the switch exists. It is
the same wedge seen along the other axis, and Q4c asks you to say so.

**Q4a.** Write `count_fixed_points(a1, a2, beta, gamma)` returning the total
number of fixed points (stable and saddle together) of the asymmetric toggle.
`posb.stability_report` with `grid=(1e-3, 300, 9)` is the tool — the pTAK117
states reach $u = 156$, so the default guess grid is too small.

In [ ]:
def count_fixed_points(a1, a2, beta, gamma):
    """Number of fixed points of the asymmetric toggle."""
    ...


print("pTAK117:", count_fixed_points(156.25, 15.6, 2.5, 1.0), "fixed points")

In [ ]:
grader.check("q4a")

**Q4b.** Write `bistable_band(a1, beta, gamma, a2_grid)` returning
`(lo, hi)`: the smallest and largest values in `a2_grid` at which the toggle
has three fixed points, or `(nan, nan)` if it never does. Sweep
`a2_grid = np.geomspace(1, 200, 120)` for pTAK117's $\alpha_1$, $\beta$,
$\gamma$, and then **draw the bifurcation diagram**: $v^*$ of every fixed point
against $\alpha_2$, stable ones as dots and the saddle as open circles, both
axes log.

In [ ]:
def bistable_band(a1, beta, gamma, a2_grid):
    """(lowest, highest) alpha_2 in the grid with three fixed points, else (nan, nan)."""
    ...


a2_grid = np.geomspace(1, 200, 120)
lo, hi = bistable_band(156.25, 2.5, 1.0, a2_grid)
print(f"pTAK117 at alpha_1 = 156.25 is bistable for alpha_2 in ({lo:.1f}, {hi:.1f}); it has 15.6")

plt.figure(figsize=(6, 4))
for a2 in a2_grid:
    for f in stability_report(toggle_model(156.25, a2, n=1.0, m=2.5), grid=(1e-3, 300, 9)):
        stable = f["type"].startswith("stable")
        plt.plot(a2, f["point"]["v"], "o", ms=4, mfc="k" if stable else "none", mec="k")
plt.axvline(15.6, ls=":", color="r")
plt.xscale("log"); plt.yscale("log"); plt.xlabel("alpha_2"); plt.ylabel("v*  (cI)");

In [ ]:
grader.check("q4b")

<!-- BEGIN QUESTION -->

**Q4c.** *(written)*

1. An RBS swap in front of one repressor moves **which** of $\alpha_1$,
   $\alpha_2$, $\beta$, $\gamma$, and why only that one?
2. Read your bifurcation diagram at the two folds. What happens to the three
   fixed points at each one — which pair meets, and what is the event called?
   Say which state the cell is left in on each side.
3. pTAK117's $\alpha_2 = 15.6$ sits a few percent above the lower fold you found. The paper
   reports (Fig. 5c) that near the switching threshold the population is
   **bimodal** — some cells high, some low, in one culture. Connect those two
   facts in two sentences.
4. Translate the paper's sentence about pIKE105 into this picture: on which
   side of which fold did it sit, which state was it stuck in, and what does a
   second RBS swap have to do to rescue it?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 5 — One ssrA tag, and the end of τ = −2

*T20, T23. This is item 4 of the session 8 handout.*

Everything in session 8 used $\tau = \mathrm{tr}\,J = -2$. That was true only
because both removal rates were scaled to 1. Tag cI, and only cI, with ssrA so
that it is removed $\delta_2$ times faster than LacI. Time is still measured in
LacI lifetimes:

$$\frac{du}{dt} = \frac{\alpha_1}{1+v^{\beta}} - u
\qquad\qquad
\frac{dv}{dt} = \frac{\alpha_2}{1+u^{\gamma}} - \delta_2\, v$$

<!-- BEGIN QUESTION -->

**Q5a.** *(written)*

1. Write the Jacobian at a fixed point $(u^*, v^*)$ in terms of $g_1 \equiv
   -\partial f/\partial v$ and $g_2 \equiv -\partial g/\partial u$ (both
   positive), and show that the fixed point is a saddle exactly when
   $$g_1\, g_2 > \delta_2 .$$
   (With $\delta_1$ kept general the condition reads $g_1 g_2 > \delta_1
   \delta_2$; say why $\delta_1 = 1$ here.)
2. Show that the **fixed points** of the tagged toggle are exactly those of
   the untagged toggle with $\alpha_2$ replaced by $\alpha_2/\delta_2$. So
   what does the tag do to the point's position in the $(\alpha_2, \alpha_1)$
   plane?
3. Is $\lambda = -1 \pm g$ still true on the diagonal of a symmetric toggle
   with one arm tagged? One sentence.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

**Q5b.** Build it. Write `tagged_toggle(a1, a2, beta, gamma, delta2)`
returning a `posb.Model` with species `["u", "v"]` — four `Reaction`s, as in
`posb.toggle_model` (read its source; it is twelve lines; `Model` and
`Reaction` are already imported by the setup cell) — with the removal
of $v$ at rate `delta2` instead of 1. Then write
`min_alpha2_for_bistability(delta2, a2_grid)` returning the smallest
$\alpha_2$ in `a2_grid` at which the tagged pTAK117 ($\alpha_1 = 156.25$,
$\beta = 2.5$, $\gamma = 1$) has three fixed points, or `nan`.

`stability_report(model, grid=(1e-3, 300, 9))` counts them, as in Q4.

In [ ]:
def tagged_toggle(a1, a2, beta, gamma, delta2):
    """The toggle with v removed delta2 times faster than u."""
    ...


def min_alpha2_for_bistability(delta2, a2_grid):
    """Smallest alpha_2 in the grid giving three fixed points for tagged pTAK117."""
    ...


a2_grid = np.geomspace(5, 200, 150)
for d2 in (1.0, 2.0, 3.0):
    n = len(stability_report(tagged_toggle(156.25, 15.6, 2.5, 1.0, d2), grid=(1e-3, 300, 9)))
    print(f"delta_2 = {d2:.0f}: pTAK117 has {n} fixed point(s); "
          f"bistable again from alpha_2 = {min_alpha2_for_bistability(d2, a2_grid):.1f}")

In [ ]:
grader.check("q5b")

<!-- BEGIN QUESTION -->

**Q5c.** *(written)* Session 9's ConcepTest answer was "tag the repressors"
to switch faster. Using Q5a and Q5b, state in no more than four sentences what
the tag buys, what it costs, and which knob from Q4 pays the cost back — with
the number for pTAK117 at $\delta_2 = 3$.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 6 — required for BioE 247, extra credit for BioE 147

**BioE 247** — part of the assignment, worth 6 of your 48
points.

**BioE 147** — optional, worth up to 6 points of extra credit on top of
42. You do not need it for full marks. It is not a harder version of the
same thing; it removes an assumption the earlier questions made, which is
where most of the interest is.

<!-- BEGIN QUESTION -->

**Q6.** Session 9 turned the IPTG knob on pTAK117 and found the threshold.
Do it yourself, then answer the question the session left open.

From the Fig. 5 legend, IPTG acts on the model by replacing $u$ in the
denominator of $dv/dt$ with $u/(1 + [\mathrm{IPTG}]/K)^{\eta}$, with
$K = 2.9618 \times 10^{-5}$ M and $\eta = 2.0015$.

1. Build the model as a `posb.Model` (the Q5b pattern), and find the
   threshold $[\mathrm{IPTG}]_c$ at which the number of fixed points drops
   from three to one, to two significant figures. Bisection on the count is
   enough. Report it in µM and compare with the jump in Fig. 5a.
2. Now sweep the **other** way: start at high IPTG and lower it to zero. Does
   the **high state** ever cease to exist on the way down? What does the
   answer mean for a cell that was pushed high — and what, therefore, is the
   second inducer for?
3. Fig. 5a's points 3a and 3b are one culture at one IPTG concentration,
   split into a high mode and a low mode. Say in two sentences what the
   deterministic model *can* say about that culture and what it *cannot*.

Put your code in the cell below the answer, and keep the written answer to
250 words.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

